# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

# Print dataset name and description
print("{}: {}".format(metadata['name'], metadata['description']))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant datasets, each record set, field, and column is referenced by a unique `@id`. Here, we will enumerate the available record sets, their fields, and columns, using their `@id` values.

In [ ]:
# Get overview of record sets (cr:RecordSet), fields (cr:Field), and columns (cr:column)

from pprint import pprint

# RecordSets are in metadata['recordSet'], which is a list of dicts
record_sets = metadata.get('recordSet', [])

print(f"Found {len(record_sets)} record set(s):\n")

for record_set in record_sets:
    rs_id = record_set['@id']
    print(f"RecordSet @id: {rs_id}")
    print(f"  Name: {record_set.get('name', '<No name>')}\n  Description: {record_set.get('description', '<None>')}")
    fields = record_set.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        # In Croissant, each field is referenced by its @id
        print(f"    - @id: {field['@id']}  | Name: {field.get('name', '<No name>')}")
        columns = field.get('column', [])
        if not isinstance(columns, list):
            columns = [columns]
        print("      Columns:")
        for col in columns:
            print(f"        * @id: {col['@id']}  | Name: {col.get('name', '<No name>')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below, we select all record sets by their `@id`.

In [ ]:
# Collect record set @ids for extraction

record_set_ids = [rs['@id'] for rs in metadata.get('recordSet', [])]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded RecordSet {record_set_id}: shape={df.shape}")
    print(f"Columns: {df.columns.tolist()}")

# Show preview of the first record set
if record_set_ids:
    first_record_set_id = record_set_ids[0]
    print(f"\nHead of RecordSet {first_record_set_id}:")
    display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. We reference fields via their `@id`.

Let's pick a numeric column such as GAD-7 or PHQ-9 scores and a categorical column such as 'level_of_education', referencing them by their `@id`. If you have the actual `@id`s from Section 2, insert them below.

In [ ]:
# Example IDs for demonstration (replace with your actual @id from the overview section)
# Assume main record set @id is 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/records'
# Numeric field @id for GAD-7 total score: 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/field/gad7_total_score'
# Group field @id for level_of_education: 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/field/level_of_education'

# Please update these values below with the actual @id from above!
record_set_id = record_set_ids[0] if record_set_ids else None

# Try to guess numeric and group field IDs from columns
df = dataframes[record_set_id]
print("Available columns:", df.columns.tolist())

# We'll pick the first column containing 'gad7' for demonstration
numeric_field_candidates = [col for col in df.columns if 'gad7' in col.lower() or 'phq9' in col.lower() or 'psq' in col.lower()]
numeric_field = numeric_field_candidates[0] if numeric_field_candidates else df.columns[0]

# For grouping, try to pick the column that is level_of_education
group_field_candidates = [col for col in df.columns if 'education' in col.lower()]
group_field = group_field_candidates[0] if group_field_candidates else df.columns[0]

# Now filter, normalize, and group
threshold = 10  # Example threshold value
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by the group_field
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame().reset_index()
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's plot the distribution of the selected numeric field, and the mean scores by education level with matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# Bar plot of mean score per education level
if group_field in df.columns:
    group_means = df.groupby(group_field)[numeric_field].mean().reset_index()
    plt.figure(figsize=(10,6))
    sns.barplot(x=group_field, y=numeric_field, data=group_means)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Kilifi County dataset was loaded using the Croissant schema and explored via `mlcroissant`.
- Data includes key mental health scores (e.g., GAD-7, PHQ-9), demographic variables, and enables slicing/grouping as shown above.
- Visualization revealed the distribution and potential group differences in mental health indicators.
- Croissant `@id` references ensure field-level traceability for AI-ready analysis.

**Note:** For deeper analysis, refer to the full data dictionary in the Croissant metadata and use `@id` referencing in all your workflows.